# EIT Numerical Analysis — Maxwell-Bloch Simulation

Simulates **Electromagnetically Induced Transparency (EIT)** in a 1-D atomic
medium by time-marching the coupled Maxwell-Bloch equations:

| Field | Equation |
|-------|----------|
| Probe field E | ∂E/∂t = Ng·P − c·∂E/∂z |
| Polarization P | ∂P/∂t = −γ₁·P + Ng·E + iΩ·S |
| Spin wave S | ∂S/∂t = −γ₂·S + iΩ\*·P |

Integration uses **4th-order Runge-Kutta** with a **4th-order finite-difference**
spatial gradient, JIT-compiled via **Numba** for speed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
import time
import warnings

# numba ≥ 0.61 required for Python 3.13 support.
# Install: pip install "numba>=0.61"
from numba import njit, prange

warnings.filterwarnings('ignore')

# Render animations inline as HTML (works in classic Notebook and JupyterLab)
plt.rcParams["animation.html"] = "jshtml"

# Set interactive=True only when %matplotlib widget (ipympl) is active
interactive = False
blit = interactive


## Physics helper functions


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Complex decay rates (absorb detunings into imaginary parts)
# ─────────────────────────────────────────────────────────────────────────────

def gamma_1(gamma, delta_1, delta_2):
    """
    Complex decay rate for the optical coherence ρ_eg.
        γ₁ = γ_eg − i(Δ₁ + Δ₂)
    Δ₁: one-photon detuning; Δ₂: two-photon detuning.
    """
    return gamma - 1j * (delta_1 + delta_2)


def gamma_2(gamma0, delta_2):
    """
    Complex decay rate for the spin-wave coherence ρ_sg.
        γ₂ = γ_sg − i·Δ₂
    """
    return gamma0 - 1j * delta_2


def group_velocity(c, Rabi, Ng):
    """
    EIT slow-light group velocity.
        vg = c / (1 + |Ng|² / |Ω|²)   ≈  c·|Ω|² / |Ng|²  in the deep-EIT limit.
    Returns 0 when the control field Ω is off.
    """
    if Rabi == 0:
        return 0.0 + 0.0j
    return c / (1 + (abs(Ng) / abs(Rabi)) ** 2)


def susceptibility(w, delta_2, gamma, gamma0, Ng, Rabi, k1, delta_1=0):
    """
    EIT susceptibility χ(ω).
    Im(χ) ∝ absorption;  Re(χ) ∝ dispersion (→ slow light).
    k1 is a coupling pre-factor proportional to atom density.
    """
    g1 = gamma  - 1j * (delta_1 + delta_2)   # γ₁  (complex)
    g2 = gamma0 - 1j * delta_2                # γ₂  (complex)
    denom = (g1 - 1j * w) - (1j * Rabi) * (1j * np.conjugate(Rabi)) / (g2 - 1j * w)
    return k1 / denom


# Vectorised versions for frequency-sweep plots
susceptibility_v = np.vectorize(susceptibility)
group_velocity_v  = np.vectorize(group_velocity)


## Simulation Parameters

All physical quantities are in **natural EIT units**:
* Length → mm  
* Time   → ns  
* Frequency / rate → GHz  
* Speed of light c = 0.3 mm/ns

Edit the block below to explore different regimes.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Physical parameters  (all quantities in mm / ns / GHz)
# ─────────────────────────────────────────────────────────────────────────────

c  = 0.3      # speed of light [mm/ns]
g  = 10       # atom-photon coupling constant [GHz]
N  = 100      # number of atoms in the ensemble

delta_1 = 0.0   # one-photon detuning [GHz]  (0 = on resonance)
delta_2 = 0.0   # two-photon detuning [GHz]  (0 = dark state; ≠0 → P grows)

decay_e   = 50 / 3
dephase_s = 5e-4
dephase_e = 0.0

gamma    = gamma_eg = 0.5 * (decay_e + dephase_e)
gamma0   = gamma_sg = 0.5 * dephase_s
gamma_es = 0.5 * (decay_e + dephase_e + dephase_s)
gamma1   = gamma_1(gamma, delta_1, delta_2)
gamma2   = gamma_2(gamma0, delta_2)

# ── Grid and time step ────────────────────────────────────────────────────────
# fast_mode=True:  dz=0.008 mm, dt=0.010 ns → CFL=0.375, ~4× fewer steps
# fast_mode=False: dz=0.004 mm, dt=0.005 ns → higher spatial resolution
fast_mode = True

dz = 0.008 if fast_mode else 0.004   # spatial step [mm]
dt = 0.010 if fast_mode else 0.005   # time step [ns]  →  CFL = c·dt/dz = 0.375

medium_length = 2.0     # EIT medium length [mm] — longer → better storage efficiency, but longer runtime
storage_time  = 1000.0   # [ns]  — dominant factor in run time; halve to halve runtime
max_amp_rabi  = 30.0    # [GHz] — larger → wider EIT window → better storage efficiency

print(f"fast_mode : {fast_mode}   (dz={dz} mm, dt={dt} ns)")
print(f"CFL       : {c*dt/dz:.3f}  ({'✓ stable' if c*dt/dz < 1 else '✗ UNSTABLE'})")


## Guide: manipulating the P-field through input parameters

The polarization P obeys  **∂P/∂t = −γ₁·P + Ng·E + iΩ·S**  with  γ₁ = γ_eg − i(Δ₁+Δ₂).

| Parameter | Variable | Effect on P |
|-----------|----------|-------------|
| Excited-state decay | `decay_e` → `gamma` | Sets the linewidth. Larger γ_eg → stronger P damping, broader EIT transparency window. |
| Ground-state dephasing | `dephase_s` → `gamma0` | Larger γ_sg → S decays faster → less iΩ·S feedback → P rises (EIT degrades). Long memory requires γ_sg → 0. |
| One-photon detuning | `delta_1` | Shifts Im(γ₁). For |Δ₁| >> γ_eg, absorption is off-peak and P grows; the signal is phase-shifted but less absorbed. |
| Two-photon detuning | `delta_2` | Breaks the EIT dark state. Any δ ≠ 0 causes P to ring at frequency δ during propagation (dark-state precession). |
| Control Rabi frequency | `max_amp_rabi` | **Main EIT knob.** Larger Ω → stronger P↔S mixing → deeper EIT (P suppressed). Setting Ω = 0 removes EIT entirely; P responds directly to E. |
| Coupling strength | `g`, `N` → `Ng = ig√N` | Larger Ng → E drives P more strongly → higher optical depth OD ∝ \|Ng\|². Also sets the compression of the pulse inside the medium. |
| Switching sharpness | `sharpness` | Faster switching → non-adiabatic; P spikes at each switch edge. Slow switching (small sharpness) preserves dark-state condition. |
| Storage duration | `storage_time` | Longer storage → more γ_sg·S decay → S delivers less to P during retrieval → lower efficiency. |

### Quick recipes

| Goal | Parameter changes |
|------|------------------|
| **Fully suppress P** (ideal EIT) | δ = 0, Δ = 0, Ω >> γ_eg, dephase_s ≈ 0 |
| **Maximise P** (absorbing medium) | Ω = 0; P tracks E directly |
| **Oscillating P pattern** (dark-state beating) | δ ≠ 0 (e.g. 0.01 GHz); P rings at frequency δ |
| **Slow P relaxation** (long memory) | dephase_s → very small, large storage_time |
| **Widen EIT window** (broader P suppression) | Increase max_amp_rabi |
| **Maximise optical depth** | Increase g or N |


## Signal and control-field pulse shapes


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Pulse-shaping helpers
# ─────────────────────────────────────────────────────────────────────────────

def time_pulse(storage_time, start_time, ts, sharpness):
    """
    Temporal Rabi envelope Ω(t): high → ramp-off (store) → ramp-on (retrieve).

    The control field is:
      • HIGH  before storage: compresses the probe into the medium
      • OFF   during storage: traps excitation as a spin wave S
      • HIGH  after  storage: retrieves S back into E

    Parameters
    ----------
    storage_time : float  — off-window duration [same units as ts]
    start_time   : float  — time when the ramp-off begins
    sharpness    : float  — switching steepness (apply 10^(-1/(s+0.3)) rescaling first)

    Returns
    -------
    1-D array, same shape as ts, with minimum shifted to 0.
    """
    offset1       = np.arctanh(0.8) + start_time
    storage_start = offset1 - np.arctanh(-0.8)
    offset2       = storage_start + storage_time - np.arctanh(-0.8)

    edge_off = (np.tanh(-(ts - offset1) * sharpness) + 1) / 2   # falling edge
    edge_on  = (np.tanh( (ts - offset2) * sharpness) + 1) / 2   # rising edge
    Rabi     = edge_off + edge_on
    return Rabi - np.min(Rabi)


## Signal (probe) pulse

Choose `pulse_type = 'gaussian'` or `'square'` below.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Probe pulse  — set pulse_type to switch between shapes
# ─────────────────────────────────────────────────────────────────────────────
pulse_type = 'gaussian'   # 'gaussian'  or  'square'

t_FWHM = 2                            # Gaussian FWHM in time [ns]
tau    = t_FWHM / 2.355               # 1/e half-width in time  [ns]
sigma  = tau * c                      # 1/e half-width in space [mm]
lim    = sigma * 4 * 2.355            # grid padding on each side of the pulse [mm]

pulse_width_outside = 2 * lim         # total spatial extent of the pulse [mm]
enter  = lim                          # z where the medium begins [mm]
exit_  = enter + medium_length        # z where the medium ends   [mm]

# Grid: left padding = lim, medium = medium_length, right padding = lim only.
# Right side needs just one lim because the retrieved pulse exits slowly; using
# lim*4 (previous default) bloated the grid 2-3x unnecessarily.
z_real = np.arange(-lim, medium_length + lim * 2, dz)
z      = z_real.astype(np.complex128)

if pulse_type == 'gaussian':
    amplitude = 10 / (sigma * (2 * np.pi) ** 0.5)
    signal    = amplitude * np.exp(-z_real**2 / (2 * sigma**2))
    label     = 'Gaussian pulse'
else:
    signal    = np.zeros(len(z_real))
    i_lo      = np.argmin(np.abs(z_real + 2))
    i_hi      = np.argmin(np.abs(z_real - 2))
    signal[i_lo:i_hi] = 1.0
    label     = 'Square pulse'

signal = signal.astype(np.complex128)

plt.figure(figsize=(9, 3))
plt.plot(z_real, signal.real, color='steelblue', label=label)
plt.axvline(enter, color='crimson', linestyle='--', lw=1.2, label='Medium start')
plt.axvline(exit_, color='crimson', linestyle=':',  lw=1.2, label='Medium end')
plt.xlabel('z  [mm]')
plt.ylabel('Amplitude  [arb.]')
plt.title(f'Initial probe pulse — {label}')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Grid points: {len(z)}  |  z range: [{z_real.min():.2f}, {z_real.max():.2f}] mm")


## Rabi control field — spatial and temporal profiles


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Atom-density coupling: Ng = i·g·√N  (imaginary so Ng·E is π/2 out of phase)
#   Non-zero only inside the medium; zero outside.
# ─────────────────────────────────────────────────────────────────────────────
Ng = 1j * g * N**0.5 * np.ones(len(z), dtype=np.complex128)

enter_idx = np.argmin(np.abs(z_real - enter))
exit_idx  = np.argmin(np.abs(z_real - exit_))
Ng[:enter_idx] = 0
Ng[exit_idx:]  = 0

# Group velocity at peak Ω (for timing estimates; propagation uses c)
vg_peak    = group_velocity(c, max_amp_rabi, 1j * g * N**0.5)
cfl        = c * dt / dz   # CFL number — must be < 1

pulse_width_inside = vg_peak * (pulse_width_outside / c)
start_time = float(
    ((pulse_width_outside / c) + (medium_length / 2 - pulse_width_inside) / vg_peak).real
)
extra_time = float((medium_length / (2 * vg_peak)).real)
run_time   = 1.5 * storage_time + start_time + extra_time

print(f"Slow-down factor  : {float((c / vg_peak).real):.1f}×")
print(f"start_time        : {start_time:.2f} ns")
print(f"run_time          : {run_time:.2f} ns")
print(f"Expected steps    : {int(run_time / dt):,}   (= run_time / dt = {run_time:.1f} / {dt})")
print(f"CFL               : {cfl:.3f}   ({'✓ stable' if cfl < 1 else '✗ UNSTABLE'})")
print()
if cfl > 0.9:
    print("⚠  CFL is close to 1 — consider reducing dt slightly.")
if cfl < 0.05:
    print(f"⚠  CFL = {cfl:.3f} is very small — dt may be unnecessarily tiny.")
    print(f"   Try dt = {dz / c * 0.4:.4f} ns  (CFL ≈ 0.4)  for a 10× speedup.")

# Temporal Rabi profile
sharpness    = 0.5   # higher → sharper switching; lower → smoother ramp
sharpness    = 10 ** (-1 / (sharpness + 0.3))

tlist        = np.arange(0, run_time, dt)
rabi_profile = max_amp_rabi * time_pulse(storage_time, start_time, tlist, sharpness)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tlist, rabi_profile, color='darkorange', lw=1.5)
axes[0].set_xlabel('time  [ns]');  axes[0].set_ylabel('Ω(t)  [GHz]')
axes[0].set_title('Control-field Rabi profile Ω(t)')

axes[1].plot(z_real, np.abs(Ng), color='steelblue', lw=1.5)
axes[1].axvline(enter, color='crimson', linestyle='--', label='Medium boundary')
axes[1].axvline(exit_, color='crimson', linestyle='--')
axes[1].set_xlabel('z  [mm]');  axes[1].set_ylabel('|Ng|  [GHz]')
axes[1].set_title('Atom-density coupling profile |Ng(z)|')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


## JIT-compiled RK4 integrator

The three fields (E, P, S) are advanced by one time step using
**classical 4th-order Runge-Kutta**.  Each right-hand-side function
maps directly to one line of the Maxwell-Bloch equations.

`dE`, `dP`, `dS` are the accumulated stage increments (0 on the first stage).


In [ ]:
@njit(fastmath=True, parallel=True)
def grad_jit(f, dz):
    """
    4th-order centred finite-difference gradient with 2nd-order boundary stencils.
    parallel=True enables multi-core execution of the interior prange loop.
    """
    g = np.zeros_like(f)
    n = len(f) - 1
    for i in prange(2, n - 1):
        g[i] = (-f[i+2] + 8*f[i+1] - 8*f[i-1] + f[i-2]) / (12 * dz)
    # Left boundary (forward 2nd-order stencil)
    g[0] = (-3*f[0] + 4*f[1] - f[2]) / (2 * dz)
    g[1] = ( f[2]   - f[0])           / (2 * dz)
    # Right boundary (backward 2nd-order stencil)
    g[n]   = (3*f[n]  - 4*f[n-1] + f[n-2]) / (2 * dz)
    g[n-1] = (f[n]    - f[n-2])             / (2 * dz)
    return g


# ── Maxwell-Bloch right-hand sides ───────────────────────────────────────────
# dE / dP / dS are RK stage increments (scalar 0 on stage 1, arrays on 2–4).

@njit(fastmath=True)
def rhs_E(E, dE, P, dP, Ng, c_light, dz):
    """∂E/∂t = Ng·(P+dP) − c·∂(E+dE)/∂z"""
    return Ng * (P + dP) - c_light * grad_jit(E + dE, dz)

@njit(fastmath=True)
def rhs_P(E, dE, P, dP, S, dS, gamma1, Ng, Rabi):
    """∂P/∂t = −γ₁·(P+dP) + Ng·(E+dE) + iΩ·(S+dS)"""
    return -gamma1 * (P + dP) + Ng * (E + dE) + 1j * Rabi * (S + dS)

@njit(fastmath=True)
def rhs_S(P, dP, S, dS, gamma2, Rabi):
    """∂S/∂t = −γ₂·(S+dS) + iΩ*·(P+dP)"""
    return -gamma2 * (S + dS) + 1j * np.conjugate(Rabi) * (P + dP)


@njit(fastmath=True)
def rk4_step(E, P, S, Ng, c_light, gamma1, gamma2, Rabi, dt, dz):
    """
    One 4th-order Runge-Kutta step for all three fields simultaneously.
    Returns (E_new, P_new, S_new) at time t + dt.
    """
    # Stage 1 — slopes at t
    k1_E = rhs_E(E, 0, P, 0, Ng, c_light, dz)
    k1_P = rhs_P(E, 0, P, 0, S, 0, gamma1, Ng, Rabi)
    k1_S = rhs_S(P, 0, S, 0, gamma2, Rabi)

    # Stage 2 — slopes at t + dt/2 (using stage-1 prediction)
    k2_E = rhs_E(E, (dt/2)*k1_E, P, (dt/2)*k1_P, Ng, c_light, dz)
    k2_P = rhs_P(E, (dt/2)*k1_E, P, (dt/2)*k1_P, S, (dt/2)*k1_S, gamma1, Ng, Rabi)
    k2_S = rhs_S(P, (dt/2)*k1_P, S, (dt/2)*k1_S, gamma2, Rabi)

    # Stage 3 — slopes at t + dt/2 (using stage-2 prediction)
    k3_E = rhs_E(E, (dt/2)*k2_E, P, (dt/2)*k2_P, Ng, c_light, dz)
    k3_P = rhs_P(E, (dt/2)*k2_E, P, (dt/2)*k2_P, S, (dt/2)*k2_S, gamma1, Ng, Rabi)
    k3_S = rhs_S(P, (dt/2)*k2_P, S, (dt/2)*k2_S, gamma2, Rabi)

    # Stage 4 — slopes at t + dt (using stage-3 prediction)
    k4_E = rhs_E(E, dt*k3_E, P, dt*k3_P, Ng, c_light, dz)
    k4_P = rhs_P(E, dt*k3_E, P, dt*k3_P, S, dt*k3_S, gamma1, Ng, Rabi)
    k4_S = rhs_S(P, dt*k3_P, S, dt*k3_S, gamma2, Rabi)

    # Weighted sum (standard RK4 weights)
    E_new = E + (dt / 6) * (k1_E + 2*k2_E + 2*k3_E + k4_E)
    P_new = P + (dt / 6) * (k1_P + 2*k2_P + 2*k3_P + k4_P)
    S_new = S + (dt / 6) * (k1_S + 2*k2_S + 2*k3_S + k4_S)
    return E_new, P_new, S_new


In [ ]:
@njit(fastmath=True)
def run_simulation(E, P, S, z, dt, Ng, rabi_profile,
                   gamma1, gamma2, run_time, c_light,
                   size, dz, n_cut_frac, interval):
    """
    Time-march the Maxwell-Bloch system and save snapshots.

    Saves BOTH the real part (signed, shows oscillations into -ve axis)
    and the absolute value |field| (magnitude envelope) as float32.
    The integration itself stays in complex128.
    """
    n_z     = len(z)
    # Real parts — can be negative; shows the actual field oscillation
    Es_r    = np.zeros(size, dtype=np.float32)
    Ps_r    = np.zeros(size, dtype=np.float32)
    Ss_r    = np.zeros(size, dtype=np.float32)
    # Magnitudes — always ≥ 0; correct for S/P during storage (phase-shifted)
    Es_a    = np.zeros(size, dtype=np.float32)
    Ps_a    = np.zeros(size, dtype=np.float32)
    Ss_a    = np.zeros(size, dtype=np.float32)
    Zs      = np.zeros(size, dtype=np.float32)
    ts      = np.zeros(size[0], dtype=np.float64)
    cut_idx = int(n_z * n_cut_frac)
    snap    = 0

    n_steps = int(run_time / dt)
    for step in range(n_steps):
        Rabi    = rabi_profile[step]
        E, P, S = rk4_step(E, P, S, Ng, c_light, gamma1, gamma2, Rabi, dt, dz)
        E[:cut_idx] = 0.0 + 0.0j

        if step % interval == 0:
            for i in range(n_z):
                Es_r[snap, i] = E[i].real
                Ps_r[snap, i] = P[i].real
                Ss_r[snap, i] = S[i].real
                Es_a[snap, i] = abs(E[i])
                Ps_a[snap, i] = abs(P[i])
                Ss_a[snap, i] = abs(S[i])
                Zs[snap, i]   = z[i].real
            ts[snap] = step * dt
            snap += 1

    return Es_r, Ps_r, Ss_r, Es_a, Ps_a, Ss_a, Zs, ts


## Run the simulation


In [ ]:
E_init = signal.copy()
P_init = np.zeros_like(E_init)
S_init = np.zeros_like(E_init)

interval   = max(5, int(run_time / dt / 1500))
n_cut_frac = 0.10
MAX_MEM_MB = 2000

n_steps     = int(run_time / dt)
n_snapshots = n_steps // interval + 1
size        = (n_snapshots, len(z))
mem_mb      = 6 * n_snapshots * len(z) * 4 / 1e6

print(f"run_time         : {run_time:.1f} ns")
print(f"Grid points      : {len(z)}")
print(f"Time steps       : {n_steps:,}")
print(f"Interval         : {interval}  (one snapshot per {interval} steps)")
print(f"Snapshots        : {n_snapshots}")
print(f"Estimated memory : {mem_mb:.1f} MB")

if mem_mb > MAX_MEM_MB:
    raise MemoryError(f"Needs {mem_mb:.0f} MB > {MAX_MEM_MB} MB. "
                      f"Set fast_mode=True or reduce storage_time.")

# ── Step 1: JIT warmup — Numba compiles on the very first call (~10-30 s) ─────
# All subsequent calls reuse the compiled code and are much faster.
print("\nStep 1/3  JIT compilation (one-time, ~10-30 s)…", end=" ", flush=True)
_sz_tiny = (2, len(z))
_rp_tiny = rabi_profile[:20] if len(rabi_profile) >= 20 else rabi_profile
t_compile = time.time()
run_simulation(E_init.copy(), P_init.copy(), S_init.copy(), z, dt, Ng,
               _rp_tiny, gamma1, gamma2, len(_rp_tiny)*dt,
               c, _sz_tiny, dz, n_cut_frac, len(_rp_tiny))
print(f"done in {time.time()-t_compile:.1f} s")

# ── Step 2: Benchmark — JIT is now cached, timing is accurate ─────────────────
print("Step 2/3  Benchmarking (500 steps)…", end=" ", flush=True)
_sz_bm = (2, len(z))
_rp_bm = rabi_profile[:500] if len(rabi_profile) >= 500 else rabi_profile
t_bm   = time.time()
run_simulation(E_init.copy(), P_init.copy(), S_init.copy(), z, dt, Ng,
               _rp_bm, gamma1, gamma2, len(_rp_bm)*dt,
               c, _sz_bm, dz, n_cut_frac, len(_rp_bm))
secs_per_step = (time.time() - t_bm) / 500
est_secs      = secs_per_step * n_steps
print(f"{secs_per_step*1e6:.1f} µs/step  →  estimated {est_secs:.0f} s ({est_secs/60:.1f} min)")

# ── Step 3: Full simulation ────────────────────────────────────────────────────
print("Step 3/3  Running…", end=" ", flush=True)
t0 = time.time()
Es_r, Ps_r, Ss_r, Es_a, Ps_a, Ss_a, Zs, ts = run_simulation(
    E_init, P_init, S_init, z, dt, Ng,
    rabi_profile, gamma1, gamma2,
    run_time, c, size, dz, n_cut_frac, interval
)
elapsed = time.time() - t0
print(f"done in {elapsed:.1f} s  (estimated was {est_secs:.0f} s)")

mask = ~(Es_a == 0).all(axis=1)
Es_r, Ps_r, Ss_r = Es_r[mask], Ps_r[mask], Ss_r[mask]
Es_a, Ps_a, Ss_a = Es_a[mask], Ps_a[mask], Ss_a[mask]
Zs, ts = Zs[mask], ts[mask]
Es, Ps, Ss = Es_a, Ps_a, Ss_a   # backward-compat aliases
peak_vs_time = np.max(Es_a, axis=1)
print(f"Snapshots: {Es_a.shape[0]}  ×  {Es_a.shape[1]} points")


## Physical metrics

Computes time delay, optical depth (OD), **storage efficiency**, and
the 1/e coherence decay time from the simulation output.


In [ ]:
# ── Time delay ────────────────────────────────────────────────────────────────
# Compare the time of peak E at the medium entrance vs exit,
# subtract the free-space transit time to isolate the EIT-induced delay.
entry_time          = ts[np.argmax(Es[:, enter_idx])]
exit_time           = ts[np.argmax(Es[:, exit_idx])]
free_space_transit  = (exit_ - enter) / c
time_delay          = exit_time - entry_time - free_space_transit

# ── Optical depth (dimensionless) ─────────────────────────────────────────────
# OD = g²·N·L / (c·γ_eg) = <|Ng|²> · L / (c · γ_eg)
# (|Ng| = g√N; OD drives single-pass absorption outside the EIT window)
OD = float(
    np.mean(np.abs(Ng[enter_idx:exit_idx])**2) * (exit_ - enter) / (c * gamma)
)

# ── Peak amplitude over time ──────────────────────────────────────────────────
peak_vs_time = np.max(Es, axis=1)

# ── Storage efficiency (energy ratio) ────────────────────────────────────────
# η = ∫|E_out(t)|² dt / ∫|E_in(t)|² dt
# Probe measured just inside the medium entrance (enter_idx) and exit (exit_idx).
energy_in  = np.trapezoid(Es[:, enter_idx    ]**2, ts)
energy_out = np.trapezoid(Es[:, exit_idx - 1 ]**2, ts)
storage_efficiency = float(energy_out / energy_in) if energy_in > 0 else 0.0

# Peak-based efficiency (simpler estimate, noisier for short pulses)
peak_in  = float(np.max(np.abs(Es[:, enter_idx    ])))
peak_out = float(np.max(np.abs(Es[:, exit_idx - 1 ])))
peak_efficiency = (peak_out / peak_in) ** 2 if peak_in > 0 else 0.0

# ── 1/e coherence decay time ─────────────────────────────────────────────────
# Track the E field at the z-point with largest amplitude at mid-storage,
# then find when it drops to 1/e of its peak value.
mid_store_snap = int(np.argmin(rabi_profile) / interval)
z_peak_idx     = int(np.argmax(np.abs(Es[mid_store_snap, :])))
E_at_peak_z    = Es[:, z_peak_idx]

i_peak         = int(np.argmax(E_at_peak_z))
i_store_end    = int(np.argmin(np.abs(ts - (ts[i_peak] + storage_time))))
E_slice        = E_at_peak_z[i_peak : i_store_end]
E_slice_norm   = E_slice / E_slice[0] if len(E_slice) > 0 and E_slice[0] != 0 else E_slice
i_1e           = int(np.argmin(np.abs(E_slice_norm - 1 / np.e)))
decay_time     = ts[i_peak + i_1e] - ts[i_peak]

# ── Summary ───────────────────────────────────────────────────────────────────
sep = "─" * 46
print(sep)
print(f"  Entry time             : {entry_time:.3e} ns")
print(f"  Exit time              : {exit_time:.3e} ns")
print(f"  Free-space transit     : {free_space_transit:.3e} ns")
print(f"  EIT time delay         : {time_delay:.3e} ns")
print(sep)
print(f"  Optical depth (OD)     : {OD:.4f}   [dimensionless]")
print(f"  1/e decay time         : {decay_time:.3e} ns")
print(sep)
print(f"  Storage efficiency η   : {storage_efficiency*100:.2f} %  (energy ratio)")
print(f"  Peak efficiency        : {peak_efficiency*100:.2f} %  (peak² ratio)")
print(f"  Minimum peak |E|       : {float(np.min(peak_vs_time)):.4g}")
print(sep)


## Diagnostic plots

Field intensities at three characteristic positions: just before the medium
(input), at the midpoint, and just after the medium (output).
The dashed overlay shows the Rabi control profile Ω(t).


In [ ]:
z1 = enter_idx - 2
z2 = (enter_idx + exit_idx) // 2
z3 = exit_idx - 1
z_positions = {'Input (z₁)': z1, 'Midpoint (z₂)': z2, 'Output (z₃)': z3}
colors = ['steelblue', 'tomato', 'darkorchid']
rabi_t = np.arange(0, run_time, dt)

thresh = max_amp_rabi * 0.05
store_mask = rabi_profile < thresh
t_well_start = float(rabi_t[np.argmax(store_mask)])   if store_mask.any() else 0
t_well_end   = float(rabi_t[len(rabi_t)-np.argmax(store_mask[::-1])-1]) if store_mask.any() else run_time

field_pairs  = [('E  (probe)', Es_r, Es_a),
                ('P  (polar.)', Ps_r, Ps_a),
                ('S  (spin)', Ss_r, Ss_a)]

plt.style.use('ggplot')

# ── Row 1: |field|² (magnitude, always ≥ 0) ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (fname, farr_r, farr_a) in zip(axes, field_pairs):
    for (pname, z_idx), color in zip(z_positions.items(), colors):
        ax.plot(ts, farr_a[:, z_idx]**2, lw=1.2, color=color, label=pname)
    ax.axvspan(t_well_start, t_well_end, alpha=0.07, color='navy')
    ax2 = ax.twinx(); ax2.plot(rabi_t, rabi_profile, lw=0.8, color='k',
                                linestyle='--', alpha=0.3); ax2.set_ylim(0, max_amp_rabi*4); ax2.set_yticks([])
    ax.set_title(fname); ax.set_xlabel('time [ns]'); ax.set_ylabel('|field|²')
axes[0].legend(fontsize=8)
plt.suptitle('Magnitude  |field|²  — blue shading = storage window (Ω≈0)', fontsize=11)
plt.tight_layout(); plt.show()

# ── Row 2: field.real (signed — shows oscillations into negative axis) ─────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (fname, farr_r, farr_a) in zip(axes, field_pairs):
    for (pname, z_idx), color in zip(z_positions.items(), colors):
        ax.plot(ts, farr_r[:, z_idx], lw=1.0, color=color, label=pname)
    ax.axhline(0, color='black', lw=0.6, linestyle=':')
    ax.axvspan(t_well_start, t_well_end, alpha=0.07, color='navy')
    ax2 = ax.twinx(); ax2.plot(rabi_t, rabi_profile, lw=0.8, color='k',
                                linestyle='--', alpha=0.3); ax2.set_ylim(0, max_amp_rabi*4); ax2.set_yticks([])
    ax.set_title(fname); ax.set_xlabel('time [ns]'); ax.set_ylabel('Re(field)')
axes[0].legend(fontsize=8)
plt.suptitle('Real part  Re(field)  — note: S.real ≈ 0 during storage (S is imaginary then)', fontsize=11)
plt.tight_layout(); plt.show()

# ── Zoomed: P and S magnitude during storage window ────────────────────────────
t_mask = (ts >= t_well_start) & (ts <= t_well_end)
fig, axes2 = plt.subplots(1, 2, figsize=(13, 4))
for ax, (fname, farr_r, farr_a) in zip(axes2, field_pairs[1:]):
    sv = farr_a[t_mask, z2]**2
    y_cap = max(np.percentile(sv, 98)*3, 1e-9) if sv.max() > 0 else 1.0
    for (pname, z_idx), color in zip(z_positions.items(), colors):
        ax.plot(ts, farr_a[:, z_idx]**2, lw=1.2, color=color, label=pname)
    ax.axvspan(t_well_start, t_well_end, alpha=0.10, color='navy')
    ax.set_ylim(0, y_cap); ax.set_title(f'{fname} — zoomed to storage scale')
    ax.set_xlabel('time [ns]'); ax.set_ylabel('|field|²'); ax.legend(fontsize=8)
plt.suptitle('P and S zoomed — retrieval spike clipped to reveal stored field', fontsize=10)
plt.tight_layout(); plt.show()

# ── Spatial snapshots ──────────────────────────────────────────────────────────
snap_times = {'Before storage': t_well_start*0.95,
              'Mid-storage': (t_well_start+t_well_end)/2,
              'Before retrieval': t_well_end*1.01}
fig, axes3 = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, (label, t_t) in zip(axes3, snap_times.items()):
    idx = int(np.argmin(np.abs(ts - t_t)))
    ymax = max(float(Es_a[idx].max()), float(Ss_a[idx].max()), 1e-9)
    ax.fill_between([enter, exit_], -ymax*0.1, ymax*1.1, alpha=0.07, color='green')
    ax.plot(z_real, Es_a[idx], color='steelblue', lw=1.5, label='|E|')
    ax.plot(z_real, Ss_a[idx], color='tomato',    lw=1.5, label='|S|')
    ax.plot(z_real, Ps_a[idx]*20, color='black',  lw=0.9, linestyle='--', label='|P|×20')
    ax.axvline(enter, color='green', lw=0.8, linestyle=':')
    ax.axvline(exit_, color='green', lw=0.8, linestyle=':')
    ax.set_xlabel('z [mm]'); ax.set_ylabel('|field|')
    rabi_val = rabi_profile[min(int(ts[idx]/dt), len(rabi_profile)-1)]
    ax.set_title(f'{label}  (t={ts[idx]:.1f} ns, Ω={rabi_val:.1f} GHz)')
    if ax == axes3[0]: ax.legend(fontsize=8)
plt.suptitle('Spatial profiles — green = medium', fontsize=10)
plt.tight_layout(); plt.show()


## 3-D surface plot  E(z, t)


In [ ]:
n_snap, n_z = Zs.shape
T_grid      = np.tile(ts, (n_z, 1)).T   # broadcast ts → (n_snap, n_z)

fig = plt.figure(figsize=(10, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.plot_surface(T_grid, Zs.real, Es.real, cmap='viridis',
                linewidth=0, antialiased=False, alpha=0.85)
ax.set_xlabel('time  [ns]')
ax.set_ylabel('z  [mm]')
ax.set_zlabel('E  [arb.]')
ax.set_title('Probe field E(z, t)')
plt.tight_layout()
plt.show()


## Inline animation  — E(z, t) propagation


In [ ]:
# Target ~300 frames per animation to stay well under the 20 MB embed limit.
# Formula: n_frames = len(Es_a) / anim_speed  →  anim_speed = len / 300
anim_speed = max(2, len(Es_a) // 300)

# Raise the embed limit to 50 MB in case the animation is slightly larger.
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 50   # MB

def make_anim(farr_a, farr_r, field_name, color):
    """
    Side-by-side inline animation: |field| (left) and Re(field) (right).
    Sized and frame-limited to stay under the 20 MB jshtml embed limit.
    """
    plt.close('all')
    # Smaller figure (was 13×3.5) — directly reduces per-frame PNG size
    fig, (ax_a, ax_r) = plt.subplots(1, 2, figsize=(11, 3),
                                      tight_layout=True)

    for ax in (ax_a, ax_r):
        ax.set_xlim(z_real.min(), z_real.max())
        ax.axvspan(enter, exit_, alpha=0.10, color='green')
        ax.axvline(enter, color='green', lw=0.9, linestyle='--')
        ax.axvline(exit_, color='green', lw=0.9, linestyle='--')
        ax.set_xlabel('z  [mm]')
        ax.grid(True, alpha=0.25)

    ax_a.set_ylim(0,                    float(farr_a.max()) * 1.05)
    ax_r.set_ylim(float(farr_r.min()) * 1.05, float(farr_r.max()) * 1.05)
    ax_r.axhline(0, color='black', lw=0.5, linestyle=':')

    ax_a.set_ylabel('|field|');   ax_a.set_title(f'{field_name}  |·|')
    ax_r.set_ylabel('Re(field)'); ax_r.set_title(f'{field_name}  Re(·)')

    line_a, = ax_a.plot([], [], lw=1.4, color=color)
    line_r, = ax_r.plot([], [], lw=1.4, color=color)
    txt     = ax_a.text(0.02, 0.93, '', transform=ax_a.transAxes, fontsize=8)

    def _init():
        line_a.set_data([], []); line_r.set_data([], [])
        return line_a, line_r

    def _update(frame):
        snap = frame * anim_speed
        line_a.set_data(Zs[snap], farr_a[snap])
        line_r.set_data(Zs[snap], farr_r[snap])
        txt.set_text(f't = {ts[snap]:.1f} ns')
        return line_a, line_r, txt

    n_frames = len(farr_a) // anim_speed
    return FuncAnimation(fig, _update, init_func=_init,
                         frames=n_frames, interval=20, blit=False)

print(f"anim_speed={anim_speed}  →  {len(Es_a)//anim_speed} frames per animation")

print("E-field animation:")
anim_E = make_anim(Es_a, Es_r, 'E  (probe)', 'steelblue')
display(anim_E)

print("P-field animation:")
anim_P = make_anim(Ps_a, Ps_r, 'P  (polarization)', 'tomato')
display(anim_P)

print("S-field animation:")
anim_S = make_anim(Ss_a, Ss_r, 'S  (spin wave)', 'darkorchid')
display(anim_S)


## MP4 export  *(optional — requires ffmpeg)*

Set `SAVE_MP4 = True` and ensure `ffmpeg` is on your PATH  
(or set `plt.rcParams['animation.ffmpeg_path']` to the full binary path).


In [ ]:
SAVE_MP4 = True   # ← flip to True to export

if SAVE_MP4:
    # plt.rcParams['animation.ffmpeg_path'] = r'C:\path\to\ffmpeg.exe'

    fps            = 15
    anim_speed_mp4 = max(1, len(Es_a) // (fps * 20))   # target ~20 s of video

    def _norm(arr):
        m = np.max(np.abs(arr)); return arr / m if m > 0 else arr

    Er_n = _norm(Es_r); Pr_n = _norm(Ps_r); Sr_n = _norm(Ss_r)

    writer  = animation.FFMpegWriter(fps=fps, metadata=dict(title='EIT simulation'))
    output_file = f'eit_{int(storage_time)}ns.mp4'

    fig_mp4, ax_mp4 = plt.subplots(figsize=(10, 4.5))
    ax_mp4.set_xlim(z_real.min(), z_real.max())
    ax_mp4.set_ylim(-1.3, 1.3)
    ax_mp4.set_xlabel('z  [mm]'); ax_mp4.set_ylabel('Re(field)  [normalised]')

    # Medium shading — drawn once, stays visible throughout
    ax_mp4.axvspan(enter, exit_, alpha=0.12, color='green', label='EIT medium')
    ax_mp4.axvline(enter, color='green', lw=1.0, linestyle='--')
    ax_mp4.axvline(exit_, color='green', lw=1.0, linestyle='--')
    ax_mp4.axhline(0,     color='black', lw=0.5, linestyle=':')

    line_E, = ax_mp4.plot([], [], color='steelblue',  lw=1.5, label='E (probe)')
    line_P, = ax_mp4.plot([], [], color='tomato',     lw=1.2, label='P (polariz.)')
    line_S, = ax_mp4.plot([], [], color='darkorchid', lw=1.2, label='S (spin wave)')
    time_txt = ax_mp4.text(0.02, 0.93, '', transform=ax_mp4.transAxes, fontsize=10)
    rabi_txt = ax_mp4.text(0.70, 0.93, '', transform=ax_mp4.transAxes, fontsize=9,
                           color='darkorange')
    ax_mp4.legend(loc='upper right', fontsize=9)

    z_plot = Zs[0]

    with writer.saving(fig_mp4, output_file, dpi=120):
        for i in range(len(Er_n) // anim_speed_mp4):
            idx = i * anim_speed_mp4
            line_E.set_data(z_plot, Er_n[idx])
            line_P.set_data(z_plot, Pr_n[idx])
            line_S.set_data(z_plot, Sr_n[idx])
            time_txt.set_text(f't = {ts[idx]:.1f} ns')
            rabi_val = rabi_profile[min(int(ts[idx]/dt), len(rabi_profile)-1)]
            rabi_txt.set_text(f'Ω = {rabi_val:.1f} GHz')
            writer.grab_frame()

    plt.close(fig_mp4)
    print(f"Saved '{output_file}'  ({len(Er_n)//anim_speed_mp4} frames @ {fps} fps)")


## Save simulation results


In [ ]:
# Save all field snapshots and the z/t axes for later analysis.
prefix = f'{int(storage_time)}ns'
np.save(f'{prefix}_Es.npy', Es)
np.save(f'{prefix}_Ps.npy', Ps)
np.save(f'{prefix}_Ss.npy', Ss)
np.save(f'{prefix}_ts.npy', ts)
np.save(f'{prefix}_z.npy',  Zs[0].real)   # z-axis is identical for every snapshot

total_mb = (Es.nbytes + Ps.nbytes + Ss.nbytes + ts.nbytes) / 1e6
print(f"Saved  {prefix}_{{Es, Ps, Ss, ts, z}}.npy   —   {total_mb:.1f} MB total")
